In [46]:
from scholarly import scholarly
import pandas as pd
import time
from habanero import Crossref

In [47]:
ObservationsData = pd.read_csv(r'../Transformed Data/ObservationsClean.csv', sep=';', index_col=0)
NativeData = pd.read_csv(r'../Transformed Data/NativeDataClean.csv', sep=';', index_col=0)

In [48]:
References = pd.DataFrame()
References['Reference'] = pd.concat([ObservationsData['Reference'], NativeData['Reference']], ignore_index=True)
References.drop_duplicates(inplace=True)
References.reset_index(drop=True, inplace=True)

In [33]:
def search_scholar(ref):
    try:
        search_query = scholarly.search_pubs(ref)
        result = next(search_query)
        
        title = result.get('bib', {}).get('title', '')
        authors = result.get('bib', {}).get('author', '')
        year = result.get('bib', {}).get('pub_year', '')
        journal = result.get('bib', {}).get('journal', '')
        
        return f"{authors} ({year}). {title}. {journal}"
    except Exception as e:
        return f"NOT FOUND: {ref}"


In [37]:
import requests

def search_crossref(title):
    url = "https://api.crossref.org/works"
    params = {"query.bibliographic": title, "rows": 1}
    response = requests.get(url, params=params)

    try:
        item = response.json()['message']['items'][0]
        authors = ', '.join([f"{a['family']} {a.get('given', '')}" for a in item.get('author', [])])
        year = item.get('issued', {}).get('date-parts', [[None]])[0][0]
        title = item.get('title', [''])[0]
        journal = item.get('container-title', [''])[0]
        volume = item.get('volume', '')
        pages = item.get('page', '')
        return f"{authors} ({year}). {title}. {journal}, {volume}, {pages}"
    except:
        return "NOT FOUND"

